# 第55课 · 前向传播（forward pass）拆解 — 算子（operator）节点：`__pow__`、`relu`、`tanh` 的实现与计算图（computation graph）展开

**目标**：为 `Value` 类补全减法、除法、乘幂（`__pow__`）、`tanh`、`relu`、`exp` 算子，使其能表达完整的前向计算图。

🔗 **Aurora 连接**：这些算子对应神经网络里的每一层——线性变换（`+`、`*`、`**`）加激活函数（`tanh`/`relu`），自动微分就建立在这套算子之上。这些算子未来将落到 Aurora ML 引擎（`src/aurora/ml/`，计划中模块）里。

← **上一课**　[L54 · Value 计算图](L54_value_autograd.ipynb)

> 上节课学习了 **Value 计算图**：标量自动微分：前向值 + 反向梯度，手写 add / mul 节点。  
> 本课将探讨 **Value 算子补全**。

## 开篇回调：L54 的 `Value` 还缺什么？

L54 已经把 `__add__` / `__mul__` 的梯度规则说清了，但还缺三类“节点动作”：`__pow__`、`relu`、`tanh` / `exp`。
本课只做一件事：把这些算子补齐，让同一套计算图能表达真正的神经网络前向。

## 本课剧情：为什么神经网络能用同一套代码同时学"加法"和"非线性"？

想象你在搭积木：`z = w1*x1 + w2*x2 + b`，然后 `output = tanh(z)`。这看起来只是两行数学，但训练时需要知道"改变 w1 多少，loss 会变多少"——对每个积木都要求偏导数。

关键洞察：**每个算子就是一个积木，知道自己怎么把梯度传回去**。

```
前向：out = a * b           → out.data = a.data * b.data
反向：∂L/∂a = ∂L/∂out · b  → a.grad += out.grad * b.data
     ∂L/∂b = ∂L/∂out · a  → b.grad += out.grad * a.data
```

每个算子在完成前向计算的同时，把"怎么反向传梯度"封装进一个 **_backward 闭包**（closure）——等到 `L.backward()` 被调用时，从输出节点逆向遍历 DAG，逐节点调用 `_backward()`。

**三个新算子**：

| 算子 | 前向 | 导数（解析式） |
|---|---|---|
| `__pow__(n)` | `x^n` | `n·x^{n-1}` |
| `relu(x)` | `max(0,x)` | `1 if x>0 else 0`（次梯度）|
| `tanh(x)` | `(e²ˣ-1)/(e²ˣ+1)` | `1 - tanh²(x)`（可用前向输出直接算）|

本节任务：补全这三个算子，让感知机 `z=w1x1+w2x2+b`，`output=tanh(z)` 端到端反向传播正确。

In [ ]:
import math

# 完整的 Value 类（含加法、乘法基础，本节要补全其余算子）
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # --- 已有算子 ---
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): return self * other

    def __neg__(self):   return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __truediv__(self, other):
        # 注意：依赖 __pow__。请先在下方练习格实现 __pow__，否则此处会抛出 NotImplementedError
        return self * other**-1
    def __rtruediv__(self, other):
        # 同上，依赖 __pow__
        return Value(other) * self**-1

    # --- 待实现算子（占位，下面编码任务格会替换）---
    def __pow__(self, n): raise NotImplementedError
    def relu(self):       raise NotImplementedError
    def tanh(self):       raise NotImplementedError
    def exp(self):        raise NotImplementedError

    # --- 反向传播（拓扑序） ---
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

print("Value 类已定义")

## 概念 1：每个算子 = 新节点 + _backward 闭包

调用 `a * b` 时发生两件事：

1. **前向**：`out = Value(a.data * b.data)`，挂载 `_prev = {a, b}`
2. **后向闭包**：在 `_backward` 里写入链式法则——`a.grad += b.data * out.grad`

闭包捕获了 `a`、`b`、`out` 三个活对象，调用 `.backward()` 时按拓扑逆序执行，梯度自动累积到叶节点的 `.grad`。

In [ ]:
# 演示：加法节点的 _backward
x = Value(3.0)
y = Value(4.0)
z = x + y       # z.data = 7.0
z.grad = 1.0
z._backward()   # 手动触发
print(f"x.grad = {x.grad}, y.grad = {y.grad}")  # 应该都是 1.0

## 概念 2：tanh 梯度用前向输出值直接表达

`tanh(x) = (e^{2x} - 1) / (e^{2x} + 1)`，其导数：

```
d(tanh(x))/dx = 1 - tanh(x)^2
```

实现技巧：`out.data` 就是已经算好的 `tanh(x)` 值，反向直接写：

```python
self.grad += (1 - out.data**2) * out.grad
```

这样不用再调一次 `math.tanh`，也不用存中间变量。

In [ ]:
# 验证公式：数值差分 vs 解析梯度
import math

x_val = 0.8
h = 1e-5
analytic = 1 - math.tanh(x_val)**2
numeric  = (math.tanh(x_val + h) - math.tanh(x_val - h)) / (2*h)
print(f"解析梯度: {analytic:.6f}")
print(f"数值梯度: {numeric:.6f}")
print(f"误差: {abs(analytic - numeric):.2e}")

## 概念 3：ReLU 梯度次梯度（subgradient）

`relu(x) = max(0, x)`，在 `x > 0` 时梯度为 1，`x < 0` 时梯度为 0，`x = 0` 处次梯度取 0：

```
d(relu(x))/dx = 1  if x > 0 else 0
```

反向实现：

```python
self.grad += (out.data > 0) * out.grad
```

`out.data > 0` 是布尔值，Python 里等于 `1` 或 `0`，直接相乘即可。ReLU 比 tanh 快，但 `x < 0` 区间梯度死亡（Dead ReLU 问题，ELU/Leaky ReLU 是常见修复方案）。

In [ ]:
# 演示 ReLU 次梯度
vals = [-2.0, 0.0, 3.0]
for v in vals:
    grad = float(v > 0)
    print(f"relu'({v:+.1f}) = {grad}")

## 1. ✏️ 实现 `Value.__pow__`, `Value.relu`, `Value.tanh`

**三步实现模板**（每个算子相同结构）：

```python
def __pow__(self, n):
    out = Value(self.data ** n, _children=(self,), _op=f'**{n}')
    def _backward():
        self.grad += n * (self.data ** (n-1)) * out.grad
    out._backward = _backward
    return out
```

**三个算子的 _backward 公式**：

| 算子 | `self.grad += ...` |
|---|---|
| `__pow__(n)` | `n · self.data^(n-1) · out.grad` |
| `relu` | `(self.data > 0) · out.grad` |
| `tanh` | `(1 - out.data²) · out.grad` ← 用前向输出直接计算 |

**验收标准**：
- `Value(-2).__pow__(2).data == 4.0`
- `Value(-1.0).relu().data == 0.0`，`Value(2.0).relu().data == 2.0`
- 感知机 `backward()` 后，数值差分误差 < 1e-5

In [ ]:
# ✏️ TODO：在下面完整实现四个算子，替换 Value 类里的占位 raise NotImplementedError

def __pow__(self, n):
    # ✏️ TODO
    assert isinstance(n, (int, float))
    t   = ...                        # ✏️ TODO: self.data**n
    out = Value(t, (self,), f'**{n}')  # _prev 和 _op 已预填，只需填 t
    def _backward():
        self.grad += ...    # ✏️ TODO: n * self.data**(n-1) * out.grad
    out._backward = _backward
    return out

def relu(self):
    t = ...                 # ✏️ TODO: max(0, self.data)
    out = Value(t, (self,), 'relu')
    def _backward():
        self.grad += ...    # ✏️ TODO: (t > 0) * out.grad
    out._backward = _backward
    return out

def tanh(self):
    t = ...                 # ✏️ TODO: math.tanh(self.data)
    out = Value(t, (self,), 'tanh')
    def _backward():
        self.grad += ...    # ✏️ TODO: (1 - t**2) * out.grad
    out._backward = _backward
    return out

def exp(self):
    t = ...                 # ✏️ TODO: math.exp(self.data)
    out = Value(t, (self,), 'exp')
    def _backward():
        self.grad += ...    # ✏️ TODO: t * out.grad
    out._backward = _backward
    return out

# 绑定到 Value 类
Value.__pow__ = __pow__
Value.relu    = relu
Value.tanh    = tanh
Value.exp     = exp

In [ ]:
# 检查：基本前向值
assert Value(-2.0).__pow__(2).data == 4.0,   "__pow__ 前向错误"
assert Value(-2.0).relu().data == 0.0,        "relu 前向：负数应输出 0"
assert Value( 0.5).relu().data == 0.5,        "relu 前向：正数应透传"
assert abs(Value(1.0).tanh().data - 0.7616) < 1e-4, "tanh(1.0) 应 ≈ 0.7616"
assert abs(Value(1.0).exp().data  - math.e)  < 1e-5, "exp(1.0) 应 ≈ e"

# 检查：__pow__ 反向
a = Value(3.0)
b = a**2          # b = 9, db/da = 2*3 = 6
b.backward()
assert abs(a.grad - 6.0) < 1e-9, f"__pow__ 反向梯度应为 6.0，得到 {a.grad}"

# 检查：relu 反向（正数区）
a = Value(2.0)
r = a.relu()
r.backward()
assert a.grad == 1.0, "relu 正数区反向梯度应为 1.0"

# 检查：relu 反向（负数区）
a = Value(-1.0)
r = a.relu()
r.backward()
assert a.grad == 0.0, "relu 负数区反向梯度应为 0.0"

# 检查：tanh 反向
a = Value(0.0)
t = a.tanh()       # tanh(0)=0, 梯度=1-0^2=1
t.backward()
assert abs(a.grad - 1.0) < 1e-9, "tanh(0) 反向梯度应为 1.0"

print("✅ 所有算子前向 & 反向检查通过")

## 参数实验：二维感知机前向传播

用 `Value` 算子搭建一个最小感知机：

```
z = w1*x1 + w2*x2 + b
output = tanh(z)         # 激活函数
```

参数：`w1=0.5, w2=-0.3, b=0.1, x1=1.0, x2=2.0`

**预期现象**：
- `z.data` 应为 `0.5*1 + (-0.3)*2 + 0.1 = 0.0`
- `output.data` 应为 `tanh(0.0) = 0.0`
- 调用 `output.backward()` 后，`w1.grad` 和 `w2.grad` 分别反映各自输入对输出的敏感度
- 把激活换成 `relu` 观察输出差异（`relu(0.0) = 0.0`，但梯度路径不同）

In [ ]:
# 二维感知机前向 + 反向
w1 = Value(0.5)
w2 = Value(-0.3)
b  = Value(0.1)
x1 = Value(1.0)
x2 = Value(2.0)

z      = w1*x1 + w2*x2 + b   # 线性组合
output = z.tanh()              # tanh 激活

print(f"z.data     = {z.data:.4f}      (期望 0.0)")
print(f"output.data= {output.data:.4f}  (期望 tanh(0.0) = 0.0)")

output.backward()
print(f"\n反向传播后：")
print(f"w1.grad = {w1.grad:.4f}   (= x1 * tanh'(z) = 1.0 * 1.0)")
print(f"w2.grad = {w2.grad:.4f}   (= x2 * tanh'(z) = 2.0 * 1.0)")
print(f"b.grad  = {b.grad:.4f}   (= tanh'(z) = 1.0)")

# 对比 relu 激活（新建独立输入，避免与上方 tanh 演示的梯度累积）
w1r, w2r, br = Value(0.5), Value(-0.3), Value(0.1)
x1r, x2r = Value(1.0), Value(2.0)   # ← 新建，不复用上方 x1/x2
zr = w1r*x1r + w2r*x2r + br
out_r = zr.relu()
out_r.backward()
print(f"\nrelu 激活：output={out_r.data:.4f}, w1.grad={w1r.grad:.4f}, w2.grad={w2r.grad:.4f}")
print("（relu 在 z=0 处次梯度取 0，梯度传播截断）")

## 本课收束

本节实现了 `__pow__`、`relu`、`tanh`、`exp` 四个算子，每个算子在返回新 `Value` 节点的同时封装了反向梯度闭包。这些算子的输出 `out.data` 就是下一节将要累积梯度的"局部变量"，梯度通过 `out.grad` 向上游流动。完整的算子集让 `Value` 能描述任意深度的计算图——对应 Aurora ML 引擎（`src/aurora/ml/`，计划中模块）里每一层的前向计算。下一节（**L56**）将把 `.backward()` 拆开手推：对整张多层计算图做拓扑排序，逐节点验证「梯度 = 局部梯度 × 上游梯度」的链式传播。

## ✏️ 白板挑战：算子梯度手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：`z = x**3` 在 `x=2` 处的梯度是多少？  
（导数：3x² → 3×4 = 12）

**问 2**：`tanh` 的导数为什么可以用前向输出值表达？  
（`d/dx tanh(x) = 1 - tanh²(x)`，不需要原始 x，只需 `out.data`）

**问 3**：对计算图 `L = relu(x*w + b)`，`x=1, w=2, b=-1`：
- 前向值：`z = x*w+b = ?`，`L = relu(z) = ?`
- 反向：`∂L/∂w = ?`（注意 relu 在 z>0 时梯度=1）

**问 4**：为什么 ReLU 在 `x=0` 处用次梯度 0（而不是 undefined）？  
（实践中 0 处概率为零，用 0 不影响训练稳定性）

**问 5**：`c = a**2 + b**2`，设 `a=3, b=4`，`c.backward()` 后 `a.grad` 和 `b.grad` 各是多少？  
（`∂c/∂a = 2a = 6`，`∂c/∂b = 2b = 8`）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import math

# 说明：算子还没实现时，TODO 里的 `...` 会让 Value(...) 抛 TypeError，
# 占位方法则抛 NotImplementedError——两种都按「待实现」处理，先给出文字答案。

# 问1：z=x**3, x=2 → grad=12
try:
    x = Value(2.0)
    z = x.__pow__(3)
    z.grad = 1.0
    z._backward()
    assert abs(x.grad - 12.0) < 1e-12, f"x.grad={x.grad}"
    print(f"Q1 ✅  d/dx(x³)|x=2 = {x.grad:.1f} = 3×2² = 12")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  d/dx(x³)|x=2 = 3×2²=12（__pow__ 待实现）")

# 问2：tanh 导数用前向输出
x_val = 0.8
tanh_val = math.tanh(x_val)
grad_analytic = 1 - tanh_val**2
grad_numeric  = (math.tanh(x_val+1e-5) - math.tanh(x_val-1e-5)) / (2e-5)
assert abs(grad_analytic - grad_numeric) < 1e-9
print(f"Q2 ✅  tanh'({x_val})=1-tanh²={grad_analytic:.6f}，数值差分={grad_numeric:.6f}")

# 问3：relu(x*w+b), x=1, w=2, b=-1
try:
    x, w, b = Value(1.0), Value(2.0), Value(-1.0)
    z = x*w + b       # z.data = 1.0
    L = z.relu()      # L.data = 1.0 (z>0)
    L.backward()
    assert abs(L.data - 1.0) < 1e-12
    assert abs(w.grad - 1.0) < 1e-12, f"w.grad={w.grad}"
    print(f"Q3 ✅  z=x*w+b={z.data:.1f}>0，L=relu(z)={L.data:.1f}，∂L/∂w={w.grad:.1f}=x=1")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  x=1,w=2,b=-1: z=1>0，L=relu(z)=1，∂L/∂w=x=1（relu 待实现）")

# 问4：relu 次梯度（概念验证）
relu_grads = {-1.0: 0, 0.0: 0, 1.0: 1}
for v, expected in relu_grads.items():
    got = float(v > 0)
    assert got == expected
print(f"Q4 ✅  relu 次梯度: x<0→0, x=0→0, x>0→1（次梯度，实践中安全）")

# 问5：c=a²+b², a=3, b=4
try:
    a, b = Value(3.0), Value(4.0)
    c = a**2 + b**2
    c.backward()
    assert abs(a.grad - 6.0) < 1e-12, f"a.grad={a.grad}"
    assert abs(b.grad - 8.0) < 1e-12, f"b.grad={b.grad}"
    print(f"Q5 ✅  c=a²+b²: ∂c/∂a={a.grad:.1f}=2a=6, ∂c/∂b={b.grad:.1f}=2b=8")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  c=a²+b²: ∂c/∂a=2×3=6, ∂c/∂b=2×4=8（算子待实现）")
print("\n🎉 算子梯度白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l55_review = {
    "pow_backward":            None,  # 记住 d(x^n)/dx=n·x^(n-1)，正确实现_backward？True/False
    "relu_subgradient":        None,  # 理解 relu 在 x=0 用次梯度 0 的原因？True/False
    "tanh_forward_trick":      None,  # 记住 tanh'=1-tanh²，用 out.data 直接计算？True/False
    "all_ops_implemented":     None,  # __pow__/relu/tanh 全部实现且感知机backward通过？True/False
    "whiteboard_passed":       None,  # 白板挑战 5 道手推全部完成？True/False
}

unfilled = [k for k, v in l55_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l55_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L55 全部通关！进入 L56：反向传播手推')

---

→ **下一课**　[L56 · 反向传播手推](L56_backward_pass.ipynb)

> 下节课将学习 **反向传播手推**：链式法则逐层展开，梯度 = 局部梯度 × 上游梯度。